In [1]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True) 

import jax.numpy as jnp
import numpy as np
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0
from src.verification.qutip_device import HiddenDevice
from src.verification.plots import plot_absolute_cityscape
from src.control.sequences import  build_local_drive_operators, generate_bell_state

# Load Configurations
config = ExperimentConfig.load("configs/two_qubit_CZ.yaml")
dt = config.pulse.dt_ns

# Build static JAX bases
H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=config.hardware.levels)

# Use your calibrated hardware parameters (or load them from your previous run)
cal_g = config.hardware.coupling_mhz
cal_d = config.hardware.idle_detuning_mhz
cal_a = config.hardware.anharmonicity_mhz

# Assemble the calibrated Hamiltonian drift
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)
# Build the 9D single-qubit drive operators
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=config.hardware.levels)

# Load the optimized CZ pulse from the previous calibration step
cal_pulse_rad = np.load("../configs/cal_pulse_rad.npy")
print("Loaded optimized CZ pulse from ../configs/cal_pulse_rad.npy")

# Initialize the QuTiP fridge simulation
fridge = HiddenDevice(
    true_coupling=cal_g, 
    true_detuning=cal_d, 
    true_anharm=cal_a,
    t1_ns=config.decoherence.t1_ns,
    t2_ns=config.decoherence.t2_ns
)
print("Environment and calibrated digital twinloaded successfully!")

Loaded optimized CZ pulse from ../configs/cal_pulse_rad.npy
Environment and calibrated digital twinloaded successfully!


In [2]:


rho_corr, c1, c2 = generate_bell_state(
    20.0, dt, cal_a, cal_d, cal_pulse_rad, 
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control
)

# Calculate fidelity to the target |00> and |11> populations
populations = np.abs(np.real(np.diag(rho_corr)))

print(f"Final Density Matrix (Corrected):\n{np.abs(rho_corr)}")

print(f"Duration: 20.0 ns | Tracked Phases: Q1={c1:5.2f}, Q2={c2:5.2f}")
print(f"  Populations: |00>: {populations[0]:.3f} | |01>: {populations[1]:.3f} | |10>: {populations[2]:.3f} | |11>: {populations[3]:.3f}")


# Render the final presentation plot
fig = plot_absolute_cityscape(rho_corr)
fig.show()

Final Density Matrix (Corrected):
[[0.53251495 0.01691161 0.10607401 0.48201722]
 [0.01691161 0.00053708 0.0033687  0.01530791]
 [0.10607401 0.0033687  0.02112935 0.09601515]
 [0.48201722 0.01530791 0.09601515 0.43630812]]
Duration: 20.0 ns | Tracked Phases: Q1=-0.92, Q2= 0.56
  Populations: |00>: 0.533 | |01>: 0.001 | |10>: 0.021 | |11>: 0.436


In [3]:
# Build the target Bell state in the 9D two-qutrit space
psi_full = (qt.tensor(qt.basis(3, 0), qt.basis(3, 0)) + qt.tensor(qt.basis(3, 1), qt.basis(3, 1))).unit()

# Extract the computational subspace indices [00, 01, 10, 11] -> indices [0, 1, 3, 4]
comp_indices = [0, 1, 3, 4]
psi_target_vec = psi_full.full().flatten()[comp_indices]
psi_target_vec = psi_target_vec / np.linalg.norm(psi_target_vec)

# Create target state vector and convert it into a density matrix (operator)
psi_target_qobj = qt.Qobj(psi_target_vec)
rho_target = psi_target_qobj * psi_target_qobj.dag()

# Wrap your physical matrix into a QuTiP operator
rho_phys_qobj = qt.Qobj(rho_corr)

# Print them clearly
print(f"Phases: {jnp.angle(rho_phys_qobj.full())}\n")
print(f"Absolute values: {np.abs(rho_phys_qobj.full())}\n")

# Calculate the true quantum fidelity
fidelity = qt.fidelity(rho_phys_qobj, rho_target) ** 2
print(f"Bell State Fidelity: {fidelity * 100:.2f}%")

Phases: [[-4.40316641e-18  2.95495455e-02 -1.32432926e+00 -4.54170001e-03]
 [-2.95495455e-02  1.22797396e-17 -1.35387881e+00 -3.40912455e-02]
 [ 1.32432926e+00  1.35387881e+00  1.99029452e-18  1.31978756e+00]
 [ 4.54170001e-03  3.40912455e-02 -1.31978756e+00  7.16882657e-18]]

Absolute values: [[0.53251495 0.01691161 0.10607401 0.48201722]
 [0.01691161 0.00053708 0.0033687  0.01530791]
 [0.10607401 0.0033687  0.02112935 0.09601515]
 [0.48201722 0.01530791 0.09601515 0.43630812]]

Bell State Fidelity: 96.64%
